# Crown Tracking Pipeline (Multithreshold + Consensus)

This notebook runs the complete crown tracking workflow using the reusable `TreeTrackingGraph` class in `src/flask_app_tracking/tree_tracking.py`.


In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

import geopandas as gpd

ROOT = Path.cwd().resolve().parents[1]
APP_DIR = ROOT / 'src' / 'flask_app_tracking'
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import tree_tracking
importlib.reload(tree_tracking)
from tree_tracking import TreeTrackingGraph

print('Imported TreeTrackingGraph from:', APP_DIR)

In [ ]:
# Choose dataset: 'SIT' or 'LHC'
DATASET = 'LHC'  # <- switch to 'SIT' for SIT run
RUN_TAG = DATASET.lower()

if DATASET.upper() == 'SIT':
    MULTITHRESH_DIR = ROOT / 'output' / 'detectree_om_sit_multithreshold/crowns_multithreshold'
    ORTHO_DIR = ROOT / 'input' / 'input_om_sit'
    OUTPUT_DIR = ROOT / 'output' / f'{RUN_TAG}_tracking_rerun_1Apr26'
    SIT_OM_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
    VALID_LHC_STEMS = []
elif DATASET.upper() == 'LHC':
    MULTITHRESH_DIR = ROOT / 'output' / 'detectree_om_lhc_multithreshold_smaller_tiles/crowns_multithreshold'
    ORTHO_DIR = ROOT / 'input' / 'input_om_lhc'
    OUTPUT_DIR = ROOT / 'output' / f'{RUN_TAG}_tracking_rerun_1Apr26'

    # Chronological LHC sequence (excluding 9/12/25 due to gross misalignment)
    VALID_LHC_STEMS = [
        'odm_orthophoto25_10_25',
        'odm_orthophoto9_11_25',
        'odm_orthophoto20_11_25',
        'odm_orthophoto26_11_25',
        'odm_orthophoto11_1_26',
        'odm_orthophoto4_02_26',
        'odm_orthophoto20_02_26',
        'odm_orthophoto7_03_26',
    ]
    SIT_OM_IDS = []
else:
    raise ValueError(f'Unknown DATASET: {DATASET!r}. Use "SIT" or "LHC".')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATASET:', DATASET)
print('MULTITHRESH_DIR:', MULTITHRESH_DIR)
print('ORTHO_DIR:', ORTHO_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Sequence length:', len(SIT_OM_IDS) if DATASET.upper() == 'SIT' else len(VALID_LHC_STEMS))

In [ ]:
pairs = []
missing = []

if DATASET.upper() == 'SIT':
    for om_num in SIT_OM_IDS:
        gpkg = MULTITHRESH_DIR / f'OM{om_num}_multithreshold.gpkg'
        tif = ORTHO_DIR / f'sit_om{om_num}.tif'
        stem = f'sit_om{om_num}'
        if gpkg.exists() and tif.exists():
            pairs.append((str(gpkg), str(tif), stem))
        else:
            missing.append({
                'stem': stem,
                'gpkg_exists': gpkg.exists(),
                'tif_exists': tif.exists(),
            })
else:
    for stem in VALID_LHC_STEMS:
        gpkg = MULTITHRESH_DIR / f'{stem}_multithreshold.gpkg'
        tif = ORTHO_DIR / f'{stem}.tif'
        if gpkg.exists() and tif.exists():
            pairs.append((str(gpkg), str(tif), stem))
        else:
            missing.append({
                'stem': stem,
                'gpkg_exists': gpkg.exists(),
                'tif_exists': tif.exists(),
            })

if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

tracker = TreeTrackingGraph(
    auto_discover=False,
    multithresh_dir=str(MULTITHRESH_DIR),
    ortho_dir=str(ORTHO_DIR),
    output_dir=str(OUTPUT_DIR),
    simplify_tol=1.0,
    resize_factor=0.1,
    max_crowns_preview=200,
 )

tracker.file_pairs = [(g, o) for g, o, _ in pairs]
tracker.om_ids = list(range(1, len(pairs) + 1))
tracker.base_threshold_tag = None

# Alignment strategy: SIT is stable with standard PCC; LHC can benefit from tiled PCC / ECC
ALIGN_METHOD = 'pcc_tiled'
ALIGN_THRESH_TAG = 'conf_0p65'

tracker.load_multithreshold_data(
    base_threshold_tag='conf_0p45',
    load_images=True,
    align=True,
    align_method=ALIGN_METHOD,
    align_threshold_tag=ALIGN_THRESH_TAG,
 )

print('Alignment method:', ALIGN_METHOD)
print('OM sequence loaded:')
for i, (_, _, stem) in enumerate(pairs, start=1):
    print(f'  OM{i}: {stem}')
print('Total crowns loaded:', sum(len(v) for v in tracker.crowns_gdfs.values()))

In [ ]:
# # Alignment debug summary (useful for pcc_tiled)
# import pandas as pd
# import numpy as np

# if 'tracker' not in globals():
#     raise RuntimeError('tracker not found. Run Cell 4 first.')

# dbg = getattr(tracker, 'alignment_debug', {}) or {}
# if not dbg:
#     print('No alignment_debug entries found (method may not populate debug).')
# else:
#     rows = []
#     for om_id in sorted(dbg.keys()):
#         d = dbg.get(om_id, {}) or {}
#         rej = d.get('rejected', {}) if isinstance(d.get('rejected', {}), dict) else {}
#         rows.append({
#             'om_id': int(om_id),
#             'pair': f'OM{om_id-1}->OM{om_id}',
#             'fallback': d.get('fallback', ''),
#             'n_valid_tiles': d.get('n_valid_tiles', np.nan),
#             'n_inliers': d.get('n_inliers', np.nan),
#             'median_error_inliers': d.get('median_error_inliers', d.get('median_error_all', np.nan)),
#             'rej_low_texture': rej.get('low_texture', np.nan),
#             'rej_high_error': rej.get('high_error', np.nan),
#             'rej_large_shift': rej.get('too_large_shift', np.nan),
#             'rej_exception': rej.get('exception', np.nan),
#         })

#     dbg_df = pd.DataFrame(rows).sort_values('om_id')
#     print('Alignment debug rows:', len(dbg_df))
#     print('Fallback counts:')
#     print(dbg_df['fallback'].fillna('').value_counts().to_string())
#     small_cols = [
#         'pair','n_valid_tiles','n_inliers','median_error_inliers','fallback',
#         'rej_low_texture','rej_high_error','rej_large_shift','rej_exception',
#     ]
#     print('\nPer-step debug (compact):')
#     print(dbg_df[small_cols].to_string(index=False))

In [ ]:
tracker.case_configs = tracker.make_strict_aligned_configs()

tracker.build_graph_conditional(
    base_max_dist=30.0,
    overlap_gate=0.10,
    min_base_similarity=0.30,
    classify_mode='balanced',
 )

quality_text, quality_metrics = tracker.quality_report()
print(quality_text)

tracker.save_text(quality_text, f'{RUN_TAG}_tracking_quality_report.txt')
tracker.save_json(quality_metrics, f'{RUN_TAG}_tracking_quality_metrics.json')

In [ ]:
hq = tracker.assemble_high_quality_chains()
clean_chains = hq['clean_chains']
extracted_backbones = hq['extracted_backbones']
all_extracted_chains = hq['all_extracted_chains']

# Partial-chain inclusion settings (historical SIT baseline: len>=5 and one-to-one ratio>=0.9)
MIN_PARTIAL_LEN = 5
MIN_PARTIAL_ONE_TO_ONE_RATIO = 0.9

consensus_source_chains = tracker.select_consensus_source_chains(
    hq,
    include_partial=True,
    min_partial_len=MIN_PARTIAL_LEN,
    min_partial_one_to_one_ratio=MIN_PARTIAL_ONE_TO_ONE_RATIO,
 )

print('High-quality chains:', len(all_extracted_chains))
print('  Clean chains:', len(clean_chains))
print('  Extracted backbones:', len(extracted_backbones))
print('Consensus source chains:', len(consensus_source_chains))
print('  Added filtered partial chains:', len(consensus_source_chains) - len(all_extracted_chains))
print(f'Filter settings: min_partial_len={MIN_PARTIAL_LEN}, min_partial_one_to_one_ratio={MIN_PARTIAL_ONE_TO_ONE_RATIO}')

In [ ]:
chains_viz_dir = OUTPUT_DIR / 'tracked_chains_visualization'
tracker.visualize_all_chains(all_extracted_chains, str(chains_viz_dir))
print('Saved chain visualizations to:', chains_viz_dir)


In [ ]:
# Generate consensus crowns + de-duplicate near-duplicates / (almost) containments

# Finalized tuning knobs
DEDUP_IOU_THRESHOLD = 0.75
DEDUP_CONTAINMENT_BUFFER = 5.0  # >0 expands polygons for a more tolerant containment test (units = CRS units)
DEDUP_TAG = f"iou{DEDUP_IOU_THRESHOLD:.2f}_buf{DEDUP_CONTAINMENT_BUFFER:.2f}".replace('.', 'p')

# Generate RAW consensus (1 row per source chain)
consensus_gdf_raw = tracker.generate_consensus_crowns(consensus_source_chains).reset_index(drop=True)
consensus_gdf_raw['source_chain_idx'] = list(range(len(consensus_gdf_raw)))

# De-dup at the consensus level
consensus_gdf, dedup_summary = tracker.deduplicate_crowns(
    consensus_gdf_raw,
    iou_threshold=DEDUP_IOU_THRESHOLD,
    drop_contained=True,
    containment_buffer=DEDUP_CONTAINMENT_BUFFER,
    min_area=0.0,
    verbose=False,
 )

# Filter the chain list so consensus crown visualizations match the cleaned crowns
kept_chain_indices = sorted({int(x) for x in consensus_gdf['source_chain_idx'].tolist()})
consensus_source_chains_cleaned = [consensus_source_chains[i] for i in kept_chain_indices]

# Persist outputs (cleaned is the canonical consensus artifact)
CONSENSUS_GPKG = OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg'
CONSENSUS_GPKG_RAW = OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}_raw.gpkg'
consensus_gdf.to_file(CONSENSUS_GPKG, driver='GPKG')
consensus_gdf_raw.to_file(CONSENSUS_GPKG_RAW, driver='GPKG')

summary = {
    'count_raw': int(len(consensus_gdf_raw)),
    'count_cleaned': int(len(consensus_gdf)),
    'dedup_summary': dedup_summary,
    'dedup_iou_threshold': float(DEDUP_IOU_THRESHOLD),
    'dedup_containment_buffer': float(DEDUP_CONTAINMENT_BUFFER),
    'dedup_tag': str(DEDUP_TAG),
    'consensus_gpkg': str(CONSENSUS_GPKG),
    'consensus_gpkg_raw': str(CONSENSUS_GPKG_RAW),
    'quality_counts': consensus_gdf['quality'].value_counts().to_dict() if 'quality' in consensus_gdf.columns else {},
    'num_consensus_source_chains': int(len(consensus_source_chains)),
    'num_consensus_source_chains_cleaned': int(len(consensus_source_chains_cleaned)),
}

# Canonical summary + parameter-tagged summary (for record)
tracker.save_json(summary, f'consensus_crowns_summary_{RUN_TAG}.json')
tracker.save_json(summary, f'consensus_crowns_summary_{RUN_TAG}_dedup_{DEDUP_TAG}.json')

print('Consensus crowns (raw -> cleaned):', len(consensus_gdf_raw), '->', len(consensus_gdf))
print('Consensus source chains (raw -> cleaned):', len(consensus_source_chains), '->', len(consensus_source_chains_cleaned))
print('Dedup params:', {'iou_threshold': DEDUP_IOU_THRESHOLD, 'containment_buffer': DEDUP_CONTAINMENT_BUFFER})
print('Dedup summary:', dedup_summary)
print('Saved cleaned consensus:', CONSENSUS_GPKG)
print('Saved raw consensus:', CONSENSUS_GPKG_RAW)

In [ ]:
# Export (CLEANED) consensus crowns as OM1-aligned GeoJSON
import rasterio

if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
    consensus_gpkg = globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')
    if not consensus_gpkg.exists():
        raise FileNotFoundError(f'Consensus file not found: {consensus_gpkg}')
    consensus_gdf = gpd.read_file(consensus_gpkg)

if consensus_gdf.empty:
    raise ValueError('Consensus GeoDataFrame is empty. Run consensus generation first.')

om1_ortho = Path(tracker.file_pairs[0][1]) if ('tracker' in globals() and tracker.file_pairs) else None
if om1_ortho is None or not om1_ortho.exists():
    raise FileNotFoundError('Could not resolve OM1 orthomosaic path from tracker.file_pairs[0].')

with rasterio.open(om1_ortho) as src:
    om1_crs = src.crs

if consensus_gdf.crs is None:
    consensus_om1 = consensus_gdf.set_crs(om1_crs, allow_override=True)
else:
    consensus_om1 = consensus_gdf.to_crs(om1_crs)

consensus_om1 = consensus_om1[
    consensus_om1.geometry.notnull() & ~consensus_om1.geometry.is_empty
] .copy()

geojson_path = OUTPUT_DIR / f'consensus_crowns_om1_phenology_{RUN_TAG}.geojson'
consensus_om1.to_file(geojson_path, driver='GeoJSON')

print('OM1 orthomosaic:', om1_ortho)
print('GeoJSON exported:', geojson_path)
print('Crown count:', len(consensus_om1))
print('Quality counts:', consensus_om1['quality'].value_counts().to_dict() if 'quality' in consensus_om1.columns else 'n/a')

In [ ]:
consensus_viz_dir = OUTPUT_DIR / 'consensus_crowns_visualization'

# Use cleaned chains (aligned with cleaned consensus crowns) when available
chains_for_viz = globals().get('consensus_source_chains_cleaned', None)
if not chains_for_viz:
    chains_for_viz = consensus_source_chains

tracker.visualize_all_consensus_chains(chains_for_viz, str(consensus_viz_dir))
print('Saved consensus visualizations to:', consensus_viz_dir)
print('Chains visualized:', len(chains_for_viz))

In [ ]:
run_summary = {
    'dataset': DATASET,
    'num_orthomosaics': len(tracker.om_ids),
    'num_graph_nodes': int(tracker.G.number_of_nodes()),
    'num_graph_edges': int(tracker.G.number_of_edges()),
    'num_high_quality_chains': int(len(all_extracted_chains)),
    'num_consensus_source_chains': int(len(consensus_source_chains)),
    'num_consensus_source_chains_cleaned': int(len(globals().get('consensus_source_chains_cleaned', []))) if 'consensus_source_chains_cleaned' in globals() else None,
    'dedup': {
        'iou_threshold': float(globals().get('DEDUP_IOU_THRESHOLD', float('nan'))),
        'containment_buffer': float(globals().get('DEDUP_CONTAINMENT_BUFFER', float('nan'))),
        'tag': str(globals().get('DEDUP_TAG', '')),
    },
    'outputs': {
        'quality_report': str(OUTPUT_DIR / f'{RUN_TAG}_tracking_quality_report.txt'),
        'quality_metrics': str(OUTPUT_DIR / f'{RUN_TAG}_tracking_quality_metrics.json'),
        'consensus_gpkg_cleaned': str(globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')),
        'consensus_gpkg_raw': str(globals().get('CONSENSUS_GPKG_RAW', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}_raw.gpkg')),
        'consensus_geojson_om1': str(OUTPUT_DIR / f'consensus_crowns_om1_phenology_{RUN_TAG}.geojson'),
        'consensus_overlay_om1': str(OUTPUT_DIR / f'consensus_overlay_om1_{RUN_TAG}.png'),
        'chain_visualizations': str(OUTPUT_DIR / 'tracked_chains_visualization'),
        'consensus_visualizations': str(OUTPUT_DIR / 'consensus_crowns_visualization'),
    },
}

print(json.dumps(run_summary, indent=2))

In [ ]:
# Crop each consensus crown geometry from every orthomosaic using alignment-corrected geometry

# This step is optional and can be expensive; keep disabled for the default end-to-end run.
RUN_CROWN_CROPS = False

import json
import imageio.v2 as imageio
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely.affinity import translate
from shapely.geometry import mapping

if not RUN_CROWN_CROPS:
    print('Skipping crown crop export (set RUN_CROWN_CROPS=True to enable).')
else:
    if 'tracker' not in globals() or 'pairs' not in globals():
        raise RuntimeError('tracker/pairs not found. Run Cells 2-4 first.')

    if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
        consensus_fp = globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')
        if not consensus_fp.exists():
            raise FileNotFoundError(f'Consensus file not found: {consensus_fp}')
        consensus_gdf = gpd.read_file(consensus_fp)

    if consensus_gdf.empty:
        raise ValueError('Consensus crowns are empty. Run consensus generation first.')

    crop_root = OUTPUT_DIR / 'consensus_crown_crops_all_oms'
    crop_root.mkdir(parents=True, exist_ok=True)

    def to_uint8_rgb(arr):
        if arr.shape[-1] > 3:
            arr = arr[..., :3]
        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        if arr.dtype == np.uint8:
            return arr
        arr = arr.astype(np.float32)
        lo = float(np.nanpercentile(arr, 2))
        hi = float(np.nanpercentile(arr, 98))
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
            hi = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else 1.0
            if hi <= lo:
                hi = lo + 1.0
        arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
        return (arr * 255).astype(np.uint8)

    manifest_rows = []

    for crown_idx, crown in consensus_gdf.reset_index(drop=True).iterrows():
        crown_id = f'crown_{crown_idx + 1:04d}'
        crown_dir = crop_root / crown_id
        crown_dir.mkdir(parents=True, exist_ok=True)
        geom_aligned = crown.geometry

        for om_id, (_, ortho_path, stem) in enumerate(pairs, start=1):
            dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
            geom_raw = translate(geom_aligned, xoff=-dx, yoff=-dy)

            out_png = crown_dir / f'OM{om_id:02d}_{stem}.png'
            status = 'ok'
            h = 0
            w = 0

            try:
                with rasterio.open(ortho_path) as src:
                    out_image, _ = mask(src, [mapping(geom_raw)], crop=True, filled=True, nodata=0)
                    rgb = np.moveaxis(out_image, 0, -1)
                    rgb_u8 = to_uint8_rgb(rgb)
                    h, w = rgb_u8.shape[:2]
                    imageio.imwrite(out_png, rgb_u8)
            except ValueError:
                status = 'no_overlap'
            except Exception as e:
                status = f'error: {str(e)}'

            manifest_rows.append({
                'crown_id': crown_id,
                'consensus_index': int(crown_idx),
                'om_id': int(om_id),
                'stem': stem,
                'shift_dx_used': float(dx),
                'shift_dy_used': float(dy),
                'status': status,
                'height_px': int(h),
                'width_px': int(w),
                'crop_path': str(out_png),
            })

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_csv = crop_root / 'consensus_crops_manifest.csv'
    manifest_json = crop_root / 'consensus_crops_manifest.json'
    manifest_df.to_csv(manifest_csv, index=False)
    with open(manifest_json, 'w') as f:
        json.dump(manifest_rows, f, indent=2)

    summary = {
        'dataset': DATASET,
        'num_consensus_crowns': int(consensus_gdf.shape[0]),
        'num_orthomosaics': int(len(pairs)),
        'expected_total_crops': int(consensus_gdf.shape[0] * len(pairs)),
        'successful_crops': int((manifest_df['status'] == 'ok').sum()),
        'failed_or_missing': int((manifest_df['status'] != 'ok').sum()),
        'manifest_csv': str(manifest_csv),
        'manifest_json': str(manifest_json),
        'output_dir': str(crop_root),
    }

    print(json.dumps(summary, indent=2))

In [ ]:
# # Build one time-series image per crown (OM1..OM8 panels in a single figure)
# import math
# import numpy as np
# import imageio.v2 as imageio
# import matplotlib.pyplot as plt
# from pathlib import Path
# from IPython.display import Image, display

# if 'manifest_df' not in globals() or manifest_df.empty:
#     manifest_path = OUTPUT_DIR / 'consensus_crown_crops_all_oms' / 'consensus_crops_manifest.csv'
#     if not manifest_path.exists():
#         raise FileNotFoundError(f'Manifest not found: {manifest_path}')
#     manifest_df = pd.read_csv(manifest_path)

# panel_root = OUTPUT_DIR / 'consensus_crown_timeseries_panels'
# panel_root.mkdir(parents=True, exist_ok=True)

# good = manifest_df[manifest_df['status'] == 'ok'].copy()
# if good.empty:
#     raise RuntimeError('No successful crops found in manifest.')

# # Keep OM order consistent
# good = good.sort_values(['crown_id', 'om_id']).reset_index(drop=True)

# panel_records = []
# for crown_id, sub in good.groupby('crown_id', sort=True):
#     sub = sub.sort_values('om_id')
#     n = len(sub)
#     fig_w = max(16, 2.8 * n)
#     fig, axes = plt.subplots(1, n, figsize=(fig_w, 3.6))
#     if n == 1:
#         axes = [axes]

#     for ax, row in zip(axes, sub.itertuples(index=False)):
#         img_path = Path(row.crop_path)
#         try:
#             img = imageio.imread(img_path)
#             ax.imshow(img)
#             ax.set_title(f'OM{int(row.om_id)}\n{row.stem}', fontsize=9)
#         except Exception:
#             ax.text(0.5, 0.5, 'read error', ha='center', va='center')
#             ax.set_title(f'OM{int(row.om_id)}\n{row.stem}', fontsize=9)
#         ax.axis('off')

#     fig.suptitle(f'{crown_id} time series', fontsize=12, y=1.04)
#     fig.tight_layout()
#     out_path = panel_root / f'{crown_id}_timeseries.png'
#     fig.savefig(out_path, dpi=160, bbox_inches='tight')
#     plt.close(fig)
#     panel_records.append({'crown_id': crown_id, 'panel_path': str(out_path), 'num_oms': int(n)})

# panel_df = pd.DataFrame(panel_records)
# panel_csv = panel_root / 'timeseries_panels_manifest.csv'
# panel_df.to_csv(panel_csv, index=False)

# print(f'Time-series panels generated: {len(panel_df)}')
# print(f'Output folder: {panel_root}')
# print(f'Manifest: {panel_csv}')

# # Preview first 3 panels
# for p in panel_df['panel_path'].head(3):
#     print(Path(p).name)
#     display(Image(filename=p, width=1800))

In [ ]:
# Save spatial overlay (CLEANED): consensus crowns over OM1
# Writes a PNG; does not display.

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from pathlib import Path

plt.ioff()

if 'tracker' not in globals() or tracker is None:
    raise RuntimeError('tracker not found. Run Cells 2–5 first.')

# Load cleaned consensus crowns if not already in memory
if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
    consensus_path = globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')
    if not Path(consensus_path).exists():
        raise FileNotFoundError(f'Consensus file not found: {consensus_path}')
    consensus_gdf = gpd.read_file(consensus_path)

if consensus_gdf.empty:
    raise ValueError('Consensus crowns are empty. Nothing to overlay.')

om1_path = Path(tracker.file_pairs[0][1]) if tracker.file_pairs and tracker.file_pairs[0][1] else None
if om1_path is None or not om1_path.exists():
    raise FileNotFoundError('Could not resolve OM1 orthomosaic path (tracker.file_pairs[0][1]).')

# Read a downsampled RGB for fast visualization
MAX_PREVIEW = 2400
with rasterio.open(om1_path) as src:
    scale = max(src.width, src.height) / MAX_PREVIEW if max(src.width, src.height) > MAX_PREVIEW else 1.0
    out_w = int(round(src.width / scale))
    out_h = int(round(src.height / scale))
    data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
    img = np.moveaxis(data, 0, -1).astype(np.float32)
    # Robust contrast stretch
    lo = np.nanpercentile(img, 2)
    hi = np.nanpercentile(img, 98)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.nanmin(img)) if np.isfinite(np.nanmin(img)) else 0.0
        hi = float(np.nanmax(img)) if np.isfinite(np.nanmax(img)) else 1.0
        if hi <= lo:
            hi = lo + 1.0
    img = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
    tr = src.transform * rasterio.transform.Affine.scale(src.width / out_w, src.height / out_h)
    # Extent for imshow in map coordinates
    left, top = (tr.c, tr.f)
    right = left + out_w * tr.a
    bottom = top + out_h * tr.e
    extent = [left, right, bottom, top]
    om1_crs = src.crs

overlay = consensus_gdf.copy()
if overlay.crs is None:
    overlay = overlay.set_crs(om1_crs, allow_override=True)
elif om1_crs is not None and overlay.crs != om1_crs:
    overlay = overlay.to_crs(om1_crs)

overlay = overlay[overlay.geometry.notnull() & ~overlay.geometry.is_empty].copy()
if overlay.empty:
    raise ValueError('Consensus crowns have no valid geometries after filtering.')

fig, ax = plt.subplots(figsize=(14, 14))
ax.imshow(img, extent=extent)
overlay.boundary.plot(ax=ax, linewidth=1.2, color='yellow', alpha=0.95)
ax.set_title(f'Consensus crowns over OM1 (CLEANED) ({DATASET}, n={len(overlay)})')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_aspect('equal', adjustable='box')

out_png = OUTPUT_DIR / f'consensus_overlay_om1_{RUN_TAG}.png'
fig.savefig(out_png, dpi=220, bbox_inches='tight')
plt.close(fig)

print('Saved cleaned overlay PNG:', out_png)

In [ ]:
# SIT: Interactive clickable consensus crowns over the LAST orthomosaic

# Clicking a crown shows its cropped appearance across ALL orthomosaics (time series).

# Requirements covered:
# 1) Uses CLEANED consensus crowns after dedup/cleanup (includes partial chains via selection settings).
# 2) On click, crops the SAME consensus geometry from every orthomosaic (n = number of OMs).

import math
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.affinity import translate
from shapely.geometry import Polygon, MultiPolygon, mapping

import matplotlib.pyplot as plt

try:
    import plotly.graph_objects as go
    from ipywidgets import Output, VBox
    from IPython.display import display, clear_output
except Exception as e:
    raise RuntimeError('This cell requires plotly + ipywidgets in the notebook environment.') from e

if DATASET.upper() != 'SIT':
    raise RuntimeError("This visualization cell is SIT-only for now. Set DATASET='SIT' and rerun above cells.")

if 'tracker' not in globals() or tracker is None:
    raise RuntimeError('tracker not found. Run Cells 2–5 first to load data + alignment.')

if 'pairs' not in globals() or not pairs:
    raise RuntimeError('pairs not found. Run Cells 3–4 first.')

# Load CLEANED consensus crowns (canonical artifact from this pipeline)
consensus_path = Path(globals().get('CONSENSUS_GPKG', OUTPUT_DIR / 'consensus_crowns_complete_all_sit.gpkg'))
if ('consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0):
    if not consensus_path.exists():
        raise FileNotFoundError(f'Cleaned consensus crowns not found: {consensus_path}')
    consensus_gdf = gpd.read_file(consensus_path)

if consensus_gdf is None or consensus_gdf.empty:
    raise RuntimeError('Consensus crowns are empty.')

num_oms = len(pairs)
last_om_id = num_oms
last_ortho_path = Path(pairs[-1][1])
if not last_ortho_path.exists():
    raise FileNotFoundError(f'Last orthomosaic not found: {last_ortho_path}')

def _to_uint8_rgb(arr: np.ndarray) -> np.ndarray:
    if arr.ndim != 3:
        raise ValueError('Expected HxWxC array')
    if arr.shape[-1] > 3:
        arr = arr[..., :3]
    if arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=-1)
    if arr.dtype == np.uint8:
        return arr
    arr = arr.astype(np.float32)
    lo = float(np.nanpercentile(arr, 2))
    hi = float(np.nanpercentile(arr, 98))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
        hi = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else 1.0
        if hi <= lo:
            hi = lo + 1.0
    arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    return (arr * 255).astype(np.uint8)

# Downsampled preview of last OM for interactive plotting
MAX_PREVIEW_PX = 1800
with rasterio.open(last_ortho_path) as src:
    scale = max(src.width, src.height) / MAX_PREVIEW_PX if max(src.width, src.height) > MAX_PREVIEW_PX else 1.0
    out_w = int(round(src.width / scale))
    out_h = int(round(src.height / scale))
    data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
    preview_img = np.moveaxis(data, 0, -1)
    preview_img = _to_uint8_rgb(preview_img)
    preview_transform = src.transform * rasterio.transform.Affine.scale(src.width / out_w, src.height / out_h)
    raster_crs = src.crs

# CRS align consensus crowns to raster
crowns = consensus_gdf.copy()
if crowns.crs is None and raster_crs is not None:
    crowns = crowns.set_crs(raster_crs, allow_override=True)
elif raster_crs is not None and crowns.crs != raster_crs:
    crowns = crowns.to_crs(raster_crs)
crowns = crowns[crowns.geometry.notnull() & ~crowns.geometry.is_empty].reset_index(drop=True)
if crowns.empty:
    raise RuntimeError('No valid geometries in cleaned consensus crowns.')

# Translate consensus geometries into LAST OM raw coordinate system
dx_last, dy_last = tracker.alignment_shifts.get(last_om_id, (0.0, 0.0))
geom_last_raw = [translate(g, xoff=-dx_last, yoff=-dy_last) for g in crowns.geometry.tolist()]

def _geom_to_xy_pix(geom) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    if geom is None:
        return None
    if isinstance(geom, Polygon):
        poly = geom
    elif isinstance(geom, MultiPolygon):
        parts = list(geom.geoms)
        if not parts:
            return None
        poly = max(parts, key=lambda p: p.area)
    else:
        return None
    simple = poly.simplify(0.5)
    xs, ys = simple.exterior.xy
    rows, cols = rasterio.transform.rowcol(preview_transform, xs, ys)
    return np.asarray(cols, dtype=np.float32), np.asarray(rows, dtype=np.float32)

# Build interactive figure
fig = go.FigureWidget()
fig.add_trace(go.Image(z=preview_img))
fig.update_layout(
    title=f'SIT consensus crowns (cleaned) on LAST OM{last_om_id} — click a crown to see its time series',
    height=850,
    margin=dict(l=10, r=10, t=60, b=10),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, autorange='reversed'),
    dragmode='pan',
    hovermode='closest',
    showlegend=False,
 )

out = Output()
display(VBox([fig, out]))

# Add crown outlines (one trace per crown so each is clickable)
traces: List[go.Scatter] = []
for i, g in enumerate(geom_last_raw):
    xy = _geom_to_xy_pix(g)
    if xy is None:
        continue
    x_pix, y_pix = xy
    tr = go.Scatter(
        x=x_pix,
        y=y_pix,
        mode='lines',
        line=dict(width=1.2, color='yellow'),
        opacity=0.85,
        customdata=[i],
        hoverinfo='text',
        text=[f'Crown {i+1}'] * len(x_pix),
    )
    fig.add_trace(tr)
    traces.append(tr)

# Cache crops per crown index (optional, speeds up repeated clicks)
_crop_cache: Dict[int, List[Optional[np.ndarray]]] = {}

def _crop_timeseries_for_crown(crown_index: int) -> List[Optional[np.ndarray]]:
    if crown_index in _crop_cache:
        return _crop_cache[crown_index]
    geom_aligned = crowns.geometry.iloc[crown_index]
    crops: List[Optional[np.ndarray]] = []
    for om_id, (_, ortho_path, _stem) in enumerate(pairs, start=1):
        dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
        geom_raw = translate(geom_aligned, xoff=-dx, yoff=-dy)
        try:
            with rasterio.open(Path(ortho_path)) as src:
                out_image, _ = mask(src, [mapping(geom_raw)], crop=True, filled=True, nodata=0)
                rgb = np.moveaxis(out_image, 0, -1)
                crops.append(_to_uint8_rgb(rgb))
        except ValueError:
            crops.append(None)
        except Exception:
            crops.append(None)
    _crop_cache[crown_index] = crops
    return crops

def _show_crown_timeseries(crown_index: int) -> None:
    crops = _crop_timeseries_for_crown(crown_index)
    cols = min(7, len(crops))
    rows = int(math.ceil(len(crops) / cols))
    fig_w = 3.1 * cols
    fig_h = 3.0 * rows
    fig_ts, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))
    axes = np.asarray(axes).reshape(rows, cols)

    for idx, ax in enumerate(axes.ravel()):
        if idx >= len(crops):
            ax.axis('off')
            continue
        om_id = idx + 1
        stem = pairs[idx][2]
        crop = crops[idx]
        ax.set_title(f'OM{om_id:02d}\n{stem}', fontsize=9)
        if crop is None:
            ax.text(0.5, 0.5, 'no overlap / error', ha='center', va='center', fontsize=9)
            ax.axis('off')
            continue
        ax.imshow(crop)
        ax.axis('off')

    fig_ts.suptitle(f'Crown {crown_index+1} time series (n={len(crops)})', fontsize=12, y=1.02)
    fig_ts.tight_layout()
    plt.show()

def _render_detail_view(crown_index: int):
    with out:
        clear_output()
        print(f'Selected crown: {crown_index+1} / {len(crowns)}')
        _show_crown_timeseries(crown_index)

def _on_click(trace, points, selector):
    if points.point_inds:
        crown_index = int(trace.customdata[0])
        _render_detail_view(crown_index)

for tr in traces:
    tr.on_click(_on_click)

with out:
    clear_output()
    print(f'Loaded {len(crowns)} cleaned consensus crowns. Click an outline above to view {num_oms} crops (one per orthomosaic).')

In [ ]:
# Export a shareable, standalone HTML viewer (SIT or LHC)

# Output:
#   output/<run_tag>_tracking_rerun_1Apr26/interactive_<run_tag>_viewer/
#     index.html
#     base_underlay_*.png
#     crowns_underlay_*.geojson
#     manifest.json
#     crops/crown_XXXX/OM##_stem.png

# This is meant to be opened in a browser (no Python needed) and shared as a zip folder.

import json
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import imageio.v2 as imageio
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.affinity import translate
from shapely.geometry import Polygon, MultiPolygon, mapping

if 'DATASET' not in globals() or not isinstance(DATASET, str):
    raise RuntimeError("DATASET not set. Run Cell 3 first.")
DATASET_NAME = DATASET.upper()
if DATASET_NAME not in {'SIT', 'LHC'}:
    raise ValueError(f"Unsupported DATASET: {DATASET}")

if 'RUN_TAG' not in globals() or not isinstance(RUN_TAG, str) or not RUN_TAG:
    RUN_TAG = DATASET_NAME.lower()

if 'tracker' not in globals() or tracker is None:
    raise RuntimeError('tracker not found. Run Cell 4 first to load data + alignment.')

if 'pairs' not in globals() or not pairs:
    raise RuntimeError('pairs not found. Run Cell 4 first.')

# Load CLEANED consensus crowns from disk if needed
default_consensus = OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg'
consensus_path = Path(globals().get('CONSENSUS_GPKG', default_consensus))
if ('consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0):
    if not consensus_path.exists():
        raise FileNotFoundError(f'Cleaned consensus crowns not found: {consensus_path}')
    consensus_gdf = gpd.read_file(consensus_path)

if consensus_gdf is None or consensus_gdf.empty:
    raise RuntimeError('Consensus crowns are empty.')

EXPORT_DIR = OUTPUT_DIR / f'interactive_{RUN_TAG}_viewer'
CROPS_DIR = EXPORT_DIR / 'crops'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
CROPS_DIR.mkdir(parents=True, exist_ok=True)

# Regenerate crops even if files exist?
FORCE_REGEN_CROPS = False

num_oms = len(pairs)

# Underlay choice: use LAST OM by default (matches the Plotly SIT viewer).
# If you want OM01 instead, set UNDERLAY_OM_ID = 1.
UNDERLAY_OM_ID = num_oms
if not (1 <= UNDERLAY_OM_ID <= num_oms):
    raise ValueError(f'UNDERLAY_OM_ID must be in [1, {num_oms}]')

_underlay_gpkg, underlay_ortho_path, underlay_stem = pairs[UNDERLAY_OM_ID - 1]
underlay_ortho_path = Path(underlay_ortho_path)

def _to_uint8_rgb(arr: np.ndarray) -> np.ndarray:
    if arr.ndim != 3:
        raise ValueError('Expected HxWxC array')
    if arr.shape[-1] > 3:
        arr = arr[..., :3]
    if arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=-1)
    if arr.dtype == np.uint8:
        return arr
    arr = arr.astype(np.float32)
    lo = float(np.nanpercentile(arr, 2))
    hi = float(np.nanpercentile(arr, 98))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
        hi = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else 1.0
        if hi <= lo:
            hi = lo + 1.0
    arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    return (arr * 255).astype(np.uint8)

# 1) Write a base PNG for the chosen underlay OM (downsampled)
BASE_MAX_PX = 2600
base_png = EXPORT_DIR / f'base_underlay_OM{UNDERLAY_OM_ID:02d}_{underlay_stem}.png'

with rasterio.open(underlay_ortho_path) as src:
    scale = max(src.width, src.height) / BASE_MAX_PX if max(src.width, src.height) > BASE_MAX_PX else 1.0
    out_w = int(round(src.width / scale))
    out_h = int(round(src.height / scale))
    data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
    base_img = np.moveaxis(data, 0, -1)
    base_img = _to_uint8_rgb(base_img)
    base_transform = src.transform * rasterio.transform.Affine.scale(src.width / out_w, src.height / out_h)
    raster_crs = src.crs

imageio.imwrite(base_png, base_img)
print('DATASET:', DATASET_NAME)
print('Underlay OM:', f'OM{UNDERLAY_OM_ID:02d}', underlay_stem)
print('Wrote base image:', base_png)
print('Base image size (HxW):', base_img.shape[0], base_img.shape[1])

# 2) Build crowns GeoJSON in *pixel coordinates* for Leaflet CRS.Simple
# IMPORTANT: Leaflet CRS.Simple uses "lat" increasing upward. Raster rows increase downward.
# So we flip Y: y_leaflet = image_height - row.
img_h = int(base_img.shape[0])
img_w = int(base_img.shape[1])

crowns = consensus_gdf.copy()
if crowns.crs is None and raster_crs is not None:
    crowns = crowns.set_crs(raster_crs, allow_override=True)
elif raster_crs is not None and crowns.crs != raster_crs:
    crowns = crowns.to_crs(raster_crs)
crowns = crowns[crowns.geometry.notnull() & ~crowns.geometry.is_empty].reset_index(drop=True)

# Convert aligned consensus geometry -> raw geometry of the chosen underlay OM
dx_u, dy_u = tracker.alignment_shifts.get(UNDERLAY_OM_ID, (0.0, 0.0))
geoms_underlay_raw = [translate(g, xoff=-dx_u, yoff=-dy_u) for g in crowns.geometry.tolist()]

def _largest_polygon(geom):
    if isinstance(geom, Polygon):
        return geom
    if isinstance(geom, MultiPolygon):
        parts = list(geom.geoms)
        if not parts:
            return None
        return max(parts, key=lambda p: p.area)
    return None

def _poly_to_leaflet_coords(poly: Polygon) -> List[List[float]]:
    simple = poly.simplify(0.5)
    xs, ys = simple.exterior.xy
    rows, cols = rasterio.transform.rowcol(base_transform, xs, ys, op=float)
    coords = [[float(c), float(img_h - r)] for c, r in zip(cols, rows)]  # GeoJSON [x, y]
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])
    return coords

features = []
for i, geom in enumerate(geoms_underlay_raw):
    poly = _largest_polygon(geom)
    if poly is None:
        continue
    try:
        ring = _poly_to_leaflet_coords(poly)
    except Exception:
        continue
    features.append({
        'type': 'Feature',
        'properties': {
            'crown_index': int(i),
            'crown_label': f'crown_{i+1:04d}',
        },
        'geometry': {
            'type': 'Polygon',
            'coordinates': [ring],
        },
    })

crowns_geojson = {
    'type': 'FeatureCollection',
    'features': features,
}

geojson_path = EXPORT_DIR / f'crowns_underlay_OM{UNDERLAY_OM_ID:02d}_{underlay_stem}_pixels.geojson'
with open(geojson_path, 'w') as f:
    json.dump(crowns_geojson, f)
print('Wrote crowns GeoJSON:', geojson_path, 'features:', len(features))

# 3) Export crops for every crown across every OM (used by HTML right panel)
manifest_path = EXPORT_DIR / 'manifest.json'
aligned_geoms = crowns.geometry.tolist()
manifest: List[Dict] = []

def _ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def _crop_one(src, geom_raw) -> Optional[np.ndarray]:
    try:
        out_image, _ = mask(src, [mapping(geom_raw)], crop=True, filled=True, nodata=0)
        rgb = np.moveaxis(out_image, 0, -1)
        return _to_uint8_rgb(rgb)
    except ValueError:
        return None
    except Exception:
        return None

if manifest_path.exists() and (not FORCE_REGEN_CROPS):
    with open(manifest_path, 'r') as f:
        manifest_blob = json.load(f)
    manifest = manifest_blob.get('crowns', [])
    print('Manifest already exists; skipping crop export:', manifest_path)
else:
    print('Exporting crops... this can take a while.')
    for crown_index in range(len(aligned_geoms)):
        crown_id = f'crown_{crown_index+1:04d}'
        crown_dir = CROPS_DIR / crown_id
        _ensure_dir(crown_dir)
        manifest_entry = {
            'crown_index': int(crown_index),
            'crown_id': crown_id,
            'num_oms': int(num_oms),
            'items': [],
        }
        manifest.append(manifest_entry)

    for om_id, (_gpkg_path, ortho_path, stem) in enumerate(pairs, start=1):
        ortho_path = Path(ortho_path)
        print(f'  OM{om_id:02d}: {stem} -> {ortho_path.name}')
        dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
        with rasterio.open(ortho_path) as src:
            for crown_index, geom_aligned in enumerate(aligned_geoms):
                crown_id = f'crown_{crown_index+1:04d}'
                out_png = CROPS_DIR / crown_id / f'OM{om_id:02d}_{stem}.png'
                rel_png = out_png.relative_to(EXPORT_DIR).as_posix()

                if out_png.exists() and (not FORCE_REGEN_CROPS):
                    status = 'exists'
                else:
                    geom_raw = translate(geom_aligned, xoff=-dx, yoff=-dy)
                    crop = _crop_one(src, geom_raw)
                    if crop is None:
                        status = 'no_overlap_or_error'
                    else:
                        imageio.imwrite(out_png, crop)
                        status = 'ok'

                manifest[crown_index]['items'].append({
                    'om_id': int(om_id),
                    'stem': stem,
                    'status': status,
                    'path': rel_png,
                })

    manifest_blob = {
        'dataset': DATASET_NAME,
        'run_tag': RUN_TAG,
        'num_oms': int(num_oms),
        'num_crowns': int(len(aligned_geoms)),
        'base_image': base_png.name,
        'crowns_geojson': geojson_path.name,
        'underlay_om_id': int(UNDERLAY_OM_ID),
        'underlay_stem': underlay_stem,
        'image_h': int(img_h),
        'image_w': int(img_w),
        'crowns': manifest,
    }
    with open(manifest_path, 'w') as f:
        json.dump(manifest_blob, f)
    print('Wrote manifest:', manifest_path)

# If we skipped crop export and loaded an existing manifest, still embed updated underlay metadata
if 'manifest_blob' not in locals():
    manifest_blob = {
        'dataset': DATASET_NAME,
        'run_tag': RUN_TAG,
        'num_oms': int(num_oms),
        'num_crowns': int(len(aligned_geoms)),
        'base_image': base_png.name,
        'crowns_geojson': geojson_path.name,
        'underlay_om_id': int(UNDERLAY_OM_ID),
        'underlay_stem': underlay_stem,
        'image_h': int(img_h),
        'image_w': int(img_w),
        'crowns': manifest,
    }
    with open(manifest_path, 'w') as f:
        json.dump(manifest_blob, f)
    print('Updated manifest metadata for underlay:', manifest_path)

# 4) Write index.html (Leaflet + right-hand time series panel)
html_path = EXPORT_DIR / 'index.html'
geojson_inline = json.dumps(crowns_geojson)
manifest_inline = json.dumps(manifest_blob)

html = f'''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>{DATASET_NAME} consensus crown time series</title>
  <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
        integrity="sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY="
        crossorigin=""/>
  <style>
    html, body {{ height: 100%; margin: 0; }}
    #root {{ height: 100%; display: flex; flex-direction: row; }}
    #map {{ flex: 1 1 auto; }}
    #panel {{ width: 460px; border-left: 1px solid #ddd; padding: 10px; overflow: auto; font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }}
    #panel h2 {{ margin: 0 0 6px 0; font-size: 16px; }}
    .hint {{ color: #555; font-size: 13px; margin-bottom: 10px; }}
    .grid {{ display: grid; grid-template-columns: 1fr; gap: 10px; }}
    .item {{ border: 1px solid #eee; padding: 8px; border-radius: 6px; }}
    .meta {{ font-size: 12px; color: #333; margin-bottom: 6px; }}
    img {{ width: 100%; height: auto; display: block; background: #f6f6f6; }}
  </style>
</head>
<body>
  <div id="root">
    <div id="map"></div>
    <div id="panel">
      <h2>{DATASET_NAME} crown time series</h2>
      <div class="hint">Underlay: OM{UNDERLAY_OM_ID:02d} {underlay_stem}. Click a yellow crown outline to load its crops across all orthomosaics.</div>
      <div id="content" class="grid"></div>
    </div>
  </div>
  <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"
          integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo="
          crossorigin=""></script>
  <script>
    const BASE_IMAGE = '{base_png.name}';
    const GEOJSON = {geojson_inline};
    const MANIFEST = {manifest_inline};

    const map = L.map('map', {{
      crs: L.CRS.Simple,
      minZoom: -4,
      maxZoom: 2,
      zoomSnap: 0.25,
      attributionControl: false
    }});

    const img = new Image();
    img.onload = () => {{
      const w = img.naturalWidth;
      const h = img.naturalHeight;
      const bounds = [[0,0],[h,w]];
      L.imageOverlay(BASE_IMAGE, bounds).addTo(map);
      map.fitBounds(bounds);

      const byIndex = new Map();
      for (const c of MANIFEST.crowns) byIndex.set(c.crown_index, c);

      function renderPanel(entry) {{
        const content = document.getElementById('content');
        content.innerHTML = '';
        if (!entry) return;
        const title = document.querySelector('#panel h2');
        title.textContent = '{DATASET_NAME} ' + entry.crown_id + ' (index ' + entry.crown_index + ')';

        for (const item of entry.items) {{
          const div = document.createElement('div');
          div.className = 'item';
          const meta = document.createElement('div');
          meta.className = 'meta';
          meta.textContent = 'OM' + String(item.om_id).padStart(2,'0') + '  ' + item.stem + '  (' + item.status + ')';
          const im = document.createElement('img');
          im.loading = 'lazy';
          im.alt = meta.textContent;
          im.src = item.path;
          div.appendChild(meta);
          div.appendChild(im);
          content.appendChild(div);
        }}
      }}

      L.geoJSON(GEOJSON, {{
        style: () => ({{ color: '#ffee55', weight: 2, fillOpacity: 0.0 }}),
        onEachFeature: (feature, layer) => {{
          layer.on('click', () => {{
            const idx = feature.properties.crown_index;
            renderPanel(byIndex.get(idx));
          }});
        }}
      }}).addTo(map);
    }};
    img.src = BASE_IMAGE;
  </script>
</body>
</html>
'''

with open(html_path, 'w') as f:
    f.write(html)

print('Wrote HTML:', html_path)
print('\nTo view: double-click index.html (or right click → Open With → browser):')
print('  ', html_path)
print('\nTo share: zip this folder and send it:')
print('  ', EXPORT_DIR)

# Phenology QA (Deciduous + Leaf-shed labels)

This section visualizes the outputs written by the batch phenology pipeline under `OUTPUT_DIR/phenology/`.

It helps verify:
- `is_deciduous` (deciduous score distribution + per-tree time series)
- `phenophase` labels (`leaf_on`, `leaf_off`, `transitioning`) over the OM sequence
- which crowns were filtered as `non-tree` (if any)

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PHENO_DIR = Path(OUTPUT_DIR) / "phenology"
print("PHENO_DIR:", PHENO_DIR)
if not PHENO_DIR.exists():
    raise FileNotFoundError(f"Missing phenology outputs folder: {PHENO_DIR}\nRun the batch script first (src/utility/run_leafshed_batch.py).")

scores_path = PHENO_DIR / "leafshed_tree_scores.csv"
phases_path = PHENO_DIR / "leafshed_phenophase_by_om.csv"
features_tree_path = PHENO_DIR / "phenology_features_raw_tree_only.csv"
features_path = PHENO_DIR / "phenology_features_raw.csv"
dropped_path = PHENO_DIR / "non_tree_dropped_crowns.csv"

if not scores_path.exists() or not phases_path.exists():
    raise FileNotFoundError(f"Missing required outputs in {PHENO_DIR}: {scores_path.name}, {phases_path.name}")

scores_df = pd.read_csv(scores_path)
phases_df = pd.read_csv(phases_path)
if features_tree_path.exists():
    features_df = pd.read_csv(features_tree_path)
else:
    features_df = pd.read_csv(features_path)

dropped_df = pd.read_csv(dropped_path) if dropped_path.exists() else pd.DataFrame()

print("Loaded:")
print("-", scores_path.name, scores_df.shape)
print("-", phases_path.name, phases_df.shape)
print("- features", features_df.shape, "(tree_only)" if features_tree_path.exists() else "(raw)")
print("- non-tree dropped", len(dropped_df) if not dropped_df.empty else 0)

# OM labels for plots
om_label = {i + 1: pairs[i][2] for i in range(len(pairs))}
om_ids = sorted(om_label.keys())

In [ ]:
DECIDUOUS_SCORE_THRESHOLD = 0.85

def plot_deciduous_score_overview(scores: pd.DataFrame, *, threshold: float = DECIDUOUS_SCORE_THRESHOLD) -> None:
    s = scores.copy()
    s["deciduous_score"] = pd.to_numeric(s.get("deciduous_score"), errors="coerce")
    s = s[np.isfinite(s["deciduous_score"])].copy()
    s["is_deciduous_thr"] = s["deciduous_score"] >= float(threshold)

    n_decid = int(s["is_deciduous_thr"].sum())
    n_total = int(len(s))
    n_ever = n_total - n_decid

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(s["deciduous_score"].values, bins=30, color="#4e79a7", alpha=0.85)
    axes[0].axvline(float(threshold), color="#d62728", lw=2, alpha=0.9)
    axes[0].set_title(f"Deciduous score distribution | thr={threshold:.2f} | decid={n_decid} | ever={n_ever}")
    axes[0].set_xlabel("deciduous_score")
    axes[0].set_ylabel("# crowns")

    # Quick component view (use whichever columns exist)
    xcol = "A_veg" if "A_veg" in s.columns else None
    ycol = "A_gcc" if "A_gcc" in s.columns else None
    if xcol and ycol:
        axes[1].scatter(
            pd.to_numeric(s[xcol], errors="coerce"),
            pd.to_numeric(s[ycol], errors="coerce"),
            s=10,
            alpha=0.5,
            c=s["is_deciduous_thr"].map({True: "#d62728", False: "#2ca02c"}),
        )
        axes[1].set_xlabel(xcol)
        axes[1].set_ylabel(ycol)
        axes[1].set_title("Amplitude scatter (red=decid, green=ever)")
    else:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, "Missing A_veg/A_gcc columns", ha="center", va="center")

    fig.tight_layout()
    plt.show()

    show_cols = [c for c in [
        "chain_id",
        "deciduous_score",
        "is_deciduous_thr",
        "leaf_off_start_om",
        "full_leaf_off_om",
        "leaf_on_return_om",
    ] if c in s.columns]
    display(s.sort_values("deciduous_score", ascending=False).head(12)[show_cols])
    display(s.sort_values("deciduous_score", ascending=True).head(12)[show_cols])

plot_deciduous_score_overview(scores_df, threshold=DECIDUOUS_SCORE_THRESHOLD)

if not dropped_df.empty:
    display(dropped_df)

In [ ]:
PHENO_COLORS = {
    "leaf_on": "#2ca02c",
    "leaf_off": "#d62728",
    "transitioning": "#ff7f0e",
    "stable": "#7f7f7f",
}

# If you run cells out-of-order, fall back to 0.85.
DECIDUOUS_SCORE_THRESHOLD = float(globals().get("DECIDUOUS_SCORE_THRESHOLD", 0.85))

def _stem_ticks(om_ids_in: list[int]) -> tuple[list[int], list[str]]:
    xs = list(om_ids_in)
    labels = [om_label.get(int(oid), str(oid)) for oid in xs]
    return xs, labels

def show_crown_timeseries(crown_index: int) -> None:
    """Compatibility wrapper for older notebook states.

    If the earlier crown-viewer/tracking cell has been run, it defines
    `_show_crown_timeseries(crown_index)` which can render the crop grid.
    """
    fn = globals().get("_show_crown_timeseries", None)
    if callable(fn):
        fn(int(crown_index))
        return
    print("Crown crops not available in this kernel state.")
    print("- Option A: run the earlier viewer-export cell (defines _show_crown_timeseries)")
    print("- Option B: run the 'show_crops_from_viewer' cell and call show_crops_from_viewer(chain_id)")

def show_tree_timeseries(chain_id: int, *, show_crops: bool = False) -> None:
    chain_id = int(chain_id)
    srow = scores_df[scores_df["chain_id"].astype(int) == chain_id]
    if srow.empty:
        raise ValueError(f"chain_id not found in scores_df: {chain_id}")
    srow = srow.iloc[0]
    ds = float(srow.get("deciduous_score", np.nan))
    decid = bool(np.isfinite(ds) and (ds >= float(DECIDUOUS_SCORE_THRESHOLD)))
    leaf_off_start = srow.get("leaf_off_start_om", np.nan)
    full_leaf_off = srow.get("full_leaf_off_om", np.nan)
    leaf_on_return = srow.get("leaf_on_return_om", np.nan)

    ph = phases_df[phases_df["chain_id"].astype(int) == chain_id].copy()
    ph = ph.sort_values("om_id")
    if ph.empty:
        raise ValueError(f"No phenophase rows for chain_id: {chain_id}")

    ft = features_df[features_df["chain_id"].astype(int) == chain_id].copy()
    ft = ft.sort_values("om_id")

    fig, ax = plt.subplots(1, 1, figsize=(12, 4))
    ax.plot(ph["om_id"], ph["veg_fraction_hsv_hat"], color="#1f77b4", lw=2, label="veg_hat")
    ax.scatter(ft["om_id"], ft["veg_fraction_hsv"], color="#1f77b4", s=18, alpha=0.35, label="veg_raw")

    for state, sub in ph.groupby("phenophase"):
        color = PHENO_COLORS.get(str(state), "#333333")
        ax.scatter(sub["om_id"], sub["veg_fraction_hsv_hat"], color=color, s=40, alpha=0.8, label=f"{state}")

    for v, label, color in [
        (leaf_off_start, "leaf_off_start", "#d62728"),
        (full_leaf_off, "full_leaf_off", "#d62728"),
        (leaf_on_return, "leaf_on_return", "#2ca02c"),
    ]:
        if pd.notna(v):
            ax.axvline(float(v), color=color, lw=1.5, alpha=0.6)
            ax.text(float(v) + 0.05, ax.get_ylim()[1], label, rotation=90, va="top", fontsize=9, color=color)

    xs, labels = _stem_ticks([int(x) for x in ph["om_id"].tolist()])
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(f"chain_id={chain_id} | deciduous={decid} (thr={DECIDUOUS_SCORE_THRESHOLD:.2f}) | DS={ds:.3f}")
    ax.set_xlabel("Orthomosaic (chronological)")
    ax.set_ylabel("veg_fraction_hsv")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    fig.tight_layout()
    plt.show()

    if show_crops:
        # Preferred: show crops from exported PNGs (doesn't require tracker state).
        scfv = globals().get("show_crops_from_viewer", None)
        if callable(scfv):
            scfv(chain_id)
            return

        # Fallback: show crops from the tracking viewer if available in-kernel.
        if "consensus_gdf" in globals() and "tracker" in globals():
            gdf = consensus_gdf
            if "chain_id" in gdf.columns:
                match = gdf.reset_index(drop=True).index[gdf["chain_id"].astype(int) == chain_id].tolist()
                if match:
                    crown_index = int(match[0])
                    show_crown_timeseries(crown_index)
                else:
                    print("chain_id not found in consensus_gdf for crops display")
        else:
            print("Note: consensus_gdf/tracker not available; run earlier cells to load tracking data.")
            print("Tip: run the 'show_crops_from_viewer' cell and call show_crops_from_viewer(chain_id)")

# Quick start: show one deciduous + one evergreen example (based on thr)
_s = scores_df.copy()
_s["deciduous_score"] = pd.to_numeric(_s.get("deciduous_score"), errors="coerce")
_s = _s[np.isfinite(_s["deciduous_score"])].copy()
_s["is_deciduous_thr"] = _s["deciduous_score"] >= float(DECIDUOUS_SCORE_THRESHOLD)
_decid = _s[_s["is_deciduous_thr"]].sort_values("deciduous_score", ascending=False)
_ever = _s[~_s["is_deciduous_thr"]].sort_values("deciduous_score", ascending=True)

if len(_decid) > 0:
    example_decid_chain = int(_decid["chain_id"].iloc[0])
    show_tree_timeseries(example_decid_chain, show_crops=False)
else:
    print("No deciduous crowns under current threshold.")

if len(_ever) > 0:
    example_ever_chain = int(_ever["chain_id"].iloc[0])
    show_tree_timeseries(example_ever_chain, show_crops=False)
else:
    print("No evergreen crowns under current threshold.")

In [ ]:
# Optional: interactive pickers (separate evergreen vs deciduous)
try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display

    # If you run cells out-of-order, fall back to 0.85.
    DECIDUOUS_SCORE_THRESHOLD = float(globals().get("DECIDUOUS_SCORE_THRESHOLD", 0.85))

    _s = scores_df.copy()
    _s["deciduous_score"] = pd.to_numeric(_s.get("deciduous_score"), errors="coerce")
    _s = _s[np.isfinite(_s["deciduous_score"])].copy()
    _s["is_deciduous_thr"] = _s["deciduous_score"] >= float(DECIDUOUS_SCORE_THRESHOLD)

    decid_chain_ids = (
        _s[_s["is_deciduous_thr"]]
        .sort_values("deciduous_score", ascending=False)["chain_id"]
        .astype(int)
        .unique()
        .tolist()
    )
    ever_chain_ids = (
        _s[~_s["is_deciduous_thr"]]
        .sort_values("deciduous_score", ascending=True)["chain_id"]
        .astype(int)
        .unique()
        .tolist()
    )

    dd_decid = widgets.Dropdown(options=decid_chain_ids, value=decid_chain_ids[0] if decid_chain_ids else None, description="deciduous")
    dd_ever = widgets.Dropdown(options=ever_chain_ids, value=ever_chain_ids[0] if ever_chain_ids else None, description="evergreen")
    cb_decid = widgets.Checkbox(value=False, description="show crops")
    cb_ever = widgets.Checkbox(value=False, description="show crops")
    btn_decid = widgets.Button(description="Plot deciduous")
    btn_ever = widgets.Button(description="Plot evergreen")
    out_decid = widgets.Output()
    out_ever = widgets.Output()

    def _run_decid(_=None):
        if dd_decid.value is None:
            with out_decid:
                out_decid.clear_output(wait=True)
                print("No deciduous crowns under current threshold.")
            return
        with out_decid:
            out_decid.clear_output(wait=True)
            show_tree_timeseries(int(dd_decid.value), show_crops=bool(cb_decid.value))

    def _run_ever(_=None):
        if dd_ever.value is None:
            with out_ever:
                out_ever.clear_output(wait=True)
                print("No evergreen crowns under current threshold.")
            return
        with out_ever:
            out_ever.clear_output(wait=True)
            show_tree_timeseries(int(dd_ever.value), show_crops=bool(cb_ever.value))

    btn_decid.on_click(_run_decid)
    btn_ever.on_click(_run_ever)

    ipy_display(widgets.HTML(f"<b>Deciduous threshold:</b> DS ≥ {DECIDUOUS_SCORE_THRESHOLD:.2f}"))
    ipy_display(widgets.HBox([dd_decid, cb_decid, btn_decid]))
    ipy_display(out_decid)
    ipy_display(widgets.HBox([dd_ever, cb_ever, btn_ever]))
    ipy_display(out_ever)
except Exception as e:
    print("ipywidgets not available (that's fine). Call show_tree_timeseries(chain_id) manually.")
    print("Reason:", repr(e))

In [ ]:
# Optional: show the actual crown crops from the exported standalone viewer folder
# (uses existing PNGs; no rasterio/tracker work needed).

import math
import re
import imageio.v2 as imageio

def _extract_om_id_from_name(name: str) -> int:
    m = re.match(r"OM(\d+)_", name)
    return int(m.group(1)) if m else 9999

def show_crops_from_viewer(chain_id: int, *, max_cols: int = 6) -> None:
    chain_id = int(chain_id)
    viewer_dir = Path(OUTPUT_DIR) / f"interactive_{RUN_TAG}_viewer"
    crops_root = viewer_dir / "crops"
    if not crops_root.exists():
        raise FileNotFoundError(
            f"Missing viewer crops folder: {crops_root}\n"
            "Run the viewer export cell (near the end of this notebook) or ensure it exists."
        )
    # Map chain_id -> crown_index based on row order in the consensus crowns GPKG used by the viewer
    consensus_path = Path(OUTPUT_DIR) / f"consensus_crowns_complete_all_{RUN_TAG}.gpkg"
    if not consensus_path.exists():
        raise FileNotFoundError(f"Missing consensus crowns: {consensus_path}")
    cgdf = gpd.read_file(consensus_path).reset_index(drop=True)
    if "chain_id" not in cgdf.columns:
        raise ValueError("Consensus crowns GPKG has no chain_id column")
    matches = cgdf.index[cgdf["chain_id"].astype(int) == chain_id].tolist()
    if not matches:
        raise ValueError(f"chain_id not found in consensus crowns: {chain_id}")
    crown_index = int(matches[0])
    crown_id = f"crown_{crown_index+1:04d}"
    crown_dir = crops_root / crown_id
    if not crown_dir.exists():
        raise FileNotFoundError(f"Missing crown crops folder: {crown_dir}")

    # Pull deciduous + DS for context
    srow = scores_df[scores_df["chain_id"].astype(int) == chain_id].head(1)
    if len(srow):
        ds = float(srow["deciduous_score"].iloc[0]) if "deciduous_score" in srow.columns else float("nan")
        decid = bool(srow["is_deciduous"].iloc[0]) if "is_deciduous" in srow.columns else False
    else:
        ds = float("nan")
        decid = False

    # phenophase label per OM for annotation
    ph = phases_df[phases_df["chain_id"].astype(int) == chain_id].copy()
    ph = ph.sort_values("om_id")
    state_by_om = {int(r.om_id): str(r.phenophase) for r in ph.itertuples(index=False)}

    pngs = sorted(crown_dir.glob("OM*.png"), key=lambda p: _extract_om_id_from_name(p.name))
    if not pngs:
        raise FileNotFoundError(f"No PNGs in: {crown_dir}")

    cols = min(max_cols, len(pngs))
    rows = int(math.ceil(len(pngs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows))
    axes = np.asarray(axes).reshape(rows, cols)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(pngs):
            ax.axis("off")
            continue
        p = pngs[i]
        img = imageio.imread(p)
        om_id = _extract_om_id_from_name(p.name)
        stem = om_label.get(int(om_id), p.name.replace(".png", ""))
        state = state_by_om.get(int(om_id), "?")
        color = PHENO_COLORS.get(state, "#333333")
        ax.imshow(img)
        ax.set_title(f"OM{om_id:02d} {stem}\n{state}", fontsize=9, color=color)
        ax.axis("off")
    fig.suptitle(f"chain_id={chain_id} -> {crown_id} | deciduous={decid} | DS={ds:.3f} ({viewer_dir.name})", y=1.02)
    fig.tight_layout()
    plt.show()

# Example: show crops for the same example chain used above
show_crops_from_viewer(example_chain)